In [ ]:
# --- KONFIGURÁCIÓ ---
MODEL_NAME = "hhao/qwen2.5-coder-tools:14b"
# --------------------

import os
import subprocess
import time
import threading
import sys
import re
import datetime

print("🔧 Rendszer előkészítése (Perzisztens tárolóval)...")

# 1. PERZISZTENS MAPPA BEÁLLÍTÁSA
# Átirányítjuk az Ollamát a /kaggle/working mappába, ami megmarad újraindítás után is
PERSISTENT_DIR = "/kaggle/working/ollama_cache"
if not os.path.exists(PERSISTENT_DIR):
    os.makedirs(PERSISTENT_DIR)

# Beállítjuk a környezeti változót az Ollamának
os.environ["OLLAMA_MODELS"] = PERSISTENT_DIR

print(f"💾 Modell tárhely: {PERSISTENT_DIR}")

# 2. TELEPÍTÉSEK (Ezeket sajnos mindig futtatni kell, de gyorsak)
subprocess.run("apt-get update && apt-get install -y zstd", shell=True, stdout=subprocess.DEVNULL)
subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh", shell=True, stdout=subprocess.DEVNULL
)

if not os.path.exists("cloudflared"):
    subprocess.run(
        "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared",
        shell=True,
    )
    subprocess.run("chmod +x cloudflared", shell=True)

# 3. OLLAMA INDÍTÁSA
print("🚀 Ollama indítása...")
OLLAMA_BIN = "/usr/local/bin/ollama"
if not os.path.exists(OLLAMA_BIN):
    OLLAMA_BIN = "/usr/bin/ollama"


def run_ollama():
    env = os.environ.copy()
    env["OLLAMA_HOST"] = "127.0.0.1:11434"
    env["OLLAMA_ORIGINS"] = "*"
    # Itt is átadjuk a tárhely helyét
    env["OLLAMA_MODELS"] = PERSISTENT_DIR
    subprocess.run(
        [OLLAMA_BIN, "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )


t = threading.Thread(target=run_ollama)
t.daemon = True
t.start()

# Várakozás
server_ready = False
for _ in range(20):
    try:
        subprocess.check_call(["curl", "-s", "http://127.0.0.1:11434"], stdout=subprocess.DEVNULL)
        server_ready = True
        break
    except:
        time.sleep(1)

if not server_ready:
    print("❌ Hiba: Az Ollama szerver nem indult el.")
    sys.exit(1)

# 4. MODELL ELLENŐRZÉSE (MOST MÁR TÉNYLEG LÁTNI FOGJA A RÉGIT)
print(f"🔍 Modell keresése a gyorsítótárban: {MODEL_NAME}")
try:
    # Listázzuk a modelleket a perzisztens mappából
    result = subprocess.run([OLLAMA_BIN, "list"], capture_output=True, text=True, env=os.environ)

    if MODEL_NAME in result.stdout:
        print(f"✅ MODELL MEGTALÁLVA! Nem töltöm le újra. 🚀")
    else:
        print(
            f"⬇️ Modell nincs a cache-ben. Letöltés... (Ez most eltart egy ideig, de legközelebb nem kell!)"
        )
        subprocess.run([OLLAMA_BIN, "pull", MODEL_NAME], check=True, env=os.environ)
        print("✅ Letöltés kész és elmentve.")
except Exception as e:
    print(f"❌ HIBA: {e}")
    sys.exit(1)

# 5. TUNNEL
print("🔌 Cloudflare Tunnel...")
if os.path.exists("tunnel.log"):
    os.remove("tunnel.log")
os.system("./cloudflared tunnel --url http://localhost:11434 > tunnel.log 2>&1 &")

time.sleep(8)
found_url = None
for _ in range(30):
    if os.path.exists("tunnel.log"):
        with open("tunnel.log", "r") as f:
            content = f.read()
            match = re.search(r"(https://[a-zA-Z0-9-]+\.trycloudflare\.com)", content)
            if match:
                found_url = match.group(1)
                break
    time.sleep(1)

if found_url:
    print("\n" + "=" * 60)
    print(f"🎉 SIKER! SZERVER AKTÍV.")
    print(f"👉 Base URL: {found_url}/v1")
    print(f"👉 Model ID: {MODEL_NAME}")
    print("=" * 60)
else:
    print("❌ URL hiba. Log:")
    os.system("cat tunnel.log")

# 6. KEEP-ALIVE
print("🟢 Anti-Disconnect aktív.")
try:
    counter = 0
    while True:
        time.sleep(60)
        counter += 1
        current_time = datetime.datetime.now().strftime("%H:%M:%S")
        print(f"💓 [{current_time}] Szerver fut ({counter}. perc) - URL: {found_url}/v1")
except KeyboardInterrupt:
    print("🛑 Leállítva.")